In [1]:
import os
import httpx

# Set your key in your environment or .env file first
os.environ["KEENABLE_API_KEY"] = "keen_4yd5wfrkvg_84c5b059d5f8bef0135c8c07fa556c3347a9e75daa19239e486253ba4206d549"
API_KEY = os.environ["KEENABLE_API_KEY"]

headers = {
    "X-API-Key": API_KEY,
    "Content-Type": "application/json",
}

# 1. Run a web search
search_resp = httpx.post(
    "https://api.keenable.ai/v1/search",
    headers=headers,
    json={
        "query": "top story on Hacker News today",
        "max_results": 3,
        "mode": "realtime",  # or "pro" for deeper retrieval
    },
    timeout=20.0,
)
search_resp.raise_for_status()
results = search_resp.json()["results"]

for r in results:
    print(f"- {r['title']}\n  {r['url']}\n")

# 2. Fetch the top result as clean Markdown
top_url = results[0]["url"]
fetch_resp = httpx.get(
    "https://api.keenable.ai/v1/fetch",
    headers={"X-API-Key": API_KEY},
    params={"url": top_url, "max_chars": 2000},
    timeout=20.0,
)
fetch_resp.raise_for_status()
print(fetch_resp.json()["content"][:1000])

- Hacker News
  https://news.ycombinator.com

- Top Links | Hacker News
  https://news.ycombinator.com/best?h=24&p=1

- Hacker News
  https://hackernews.com

| [![](https://news.ycombinator.com/y18.svg)](https://news.ycombinator.com) | **[Hacker News](https://news.ycombinator.com/news)**[new](https://news.ycombinator.com/newest) \| [past](https://news.ycombinator.com/front) \| [comments](https://news.ycombinator.com/newcomments) \| [ask](https://news.ycombinator.com/ask) \| [show](https://news.ycombinator.com/show) \| [jobs](https://news.ycombinator.com/jobs) \| [submit](https://news.ycombinator.com/submit) | [login](https://news.ycombinator.com/login?goto=news) |
|---|---|---|

| 1. | | [Credit card skimming scripts using WebRTC found on popular shopping sites](https://www.infohawk.com/news/2026-11-27-black-friday-webrtc-skimmer) ([infohawk.com](https://news.ycombinator.com/from?site=infohawk.com)) |
|---|---|---|
| | | 196 points by [scanr](https://news.ycombinator.com/user?id=scanr)

In [2]:
import os
import httpx

# Load from environment (avoid hardcoding keys in notebooks you might commit)
API_KEY = os.environ["KEENABLE_API_KEY"]

PRODUCT_QUERY = "Red shirt men"
SHOPPING_SITES = ["amazon.in", "flipkart.com"]  # You can also narrow paths like "amazon.in/dp"

headers = {
    "X-API-Key": API_KEY,
    "Content-Type": "application/json",
}

def search_store(site: str, query: str, max_results: int = 3):
    resp = httpx.post(
        "https://api.keenable.ai/v1/search",
        headers=headers,
        json={
            "query": query,
            "site": site,           # Restricts results to this shopping site
            "max_results": max_results,
            "mode": "realtime",     # Faster for agent loops; use "pro" for deeper results
        },
        timeout=25.0,
    )
    resp.raise_for_status()
    return resp.json().get("results", [])

def fetch_product_page(url: str, live: bool = False):
    params = {
        "url": url,
        "max_chars": 4000,
    }
    if live:
        params["live"] = "true"     # Pulls live page instead of Keenable's indexed snapshot

    resp = httpx.get(
        "https://api.keenable.ai/v1/fetch",
        headers={"X-API-Key": API_KEY},
        params=params,
        timeout=25.0,
    )
    resp.raise_for_status()
    return resp.json()

# Run across both stores
for site in SHOPPING_SITES:
    print("=" * 60)
    print(f"STORE: {site.upper()} | QUERY: {PRODUCT_QUERY}")
    print("=" * 60)

    results = search_store(site, PRODUCT_QUERY, max_results=3)
    if not results:
        print("No results found.\n")
        continue

    for i, r in enumerate(results, start=1):
        print(f"{i}. {r.get('title')}")
        print(f"   URL: {r.get('url')}")
        print(f"   Snippet: {r.get('snippet', '')[:160]}...\n")

    # Fetch clean Markdown for the top product result
    top_url = results[0]["url"]
    print(f"--> Fetching top product page: {top_url}")
    page_data = fetch_product_page(top_url, live=False)
    print(page_data.get("content", "")[:1200])
    print("\n")

STORE: AMAZON.IN | QUERY: Red shirt men
1. Buy Allen Solly Mens Casual T- Shirt_ASKPCURGF681008_L Red at Amazon.in
   URL: https://amazon.in/Allen-Solly-Regular-T-Shirt-ASKPQRGFN94502_Red/dp/B0C7ZNQY79/ref=cs_sr_dp_4
   Snippet: Material composition
60% Cotton, 40% Polyester
Pattern
Solid
Fit type
Regular Fit
Sleeve type
Half Sleeve
Collar style
Polo Collar
Length
Standard Length
Countr...

2. Buy Overlays Collection Men's Cotton |Glory Arc Relaxed Fit | Crew Neck | 100% Cotton Regular Fit T-Shirt | Stylish and Fashionable for Men | Red | Size-XS | at Amazon.in
   URL: https://amazon.in/Overlays-Glory-Arc-Relaxed-T-Shirt/dp/B0F8BSZK9F
   Snippet: - Stylish Red Color with Classic Crew Neck Design: This bold red t-shirt features a timeless crew neck and clean silhouette, making it a versatile wardrobe esse...

3. U.S. POLO ASSN. Checked Tailored Fit Shirt for Men - 100% Cotton, Breathable & Comfortable, Full Sleeves Casual Shirt for Men - 39, Red : Amazon.in: Clothing & Accessories
   UR

In [3]:
# ==============================================================================
# BLOCK 1: IMPORTS & CONFIGURATION
# What it does:
#   Loads all required libraries and centralizes runtime settings (API keys,
#   model name, SQLite file path, and Ollama hardware options).
# How it is implemented:
#   Reads KEENABLE_API_KEY from the environment so secrets are never hardcoded.
#   Uses synchronous `httpx` + `sqlite3` + `ollama` so it runs cleanly inside
#   Jupyter Notebook without asyncio event-loop conflicts.
# ==============================================================================
import os
import json
import sqlite3
from datetime import datetime, timezone
from typing import TypedDict, List, Dict, Any

import httpx
import ollama
from langgraph.graph import StateGraph, START, END

# Make sure you ran: os.environ["KEENABLE_API_KEY"] = "your_key" in an earlier cell
KEENABLE_API_KEY = os.environ["KEENABLE_API_KEY"]
KEENABLE_BASE_URL = "https://api.keenable.ai"

MODEL_NAME = "qwen3.5:4b"          # Your local 4B model in Ollama
DB_PATH = "agent_context.db"       # Local SQLite database for context offloading

# Set to {"num_gpu": 0} if you hit the Windows CUDA crash; use {} for GPU speed
OLLAMA_OPTIONS = {}


# ==============================================================================
# BLOCK 2: SQLITE + FTS5 CONTEXT-OFFLOADING DATABASE
# What it does:
#   Creates a local SQLite database that stores large webpage Markdown under a
#   `search_title` and indexes it with FTS5 (Full-Text Search) so the 4B model
#   can retrieve only tiny, relevant chunks instead of reading entire webpages.
# How it is implemented:
#   1. `pages` table stores metadata + full Markdown.
#   2. `pages_fts` virtual table (USING fts5) mirrors the text for fast keyword
#      matching and snippet extraction via SQLite's built-in `snippet()` function.
# ==============================================================================
def init_db(db_path: str = DB_PATH) -> None:
    """Initializes the SQLite tables and FTS5 full-text search index."""
    with sqlite3.connect(db_path) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                search_title TEXT NOT NULL,
                site TEXT,
                url TEXT NOT NULL,
                title TEXT,
                content TEXT NOT NULL,
                fetched_at TEXT NOT NULL
            )
        """)
        # FTS5 virtual table enables fast keyword search across saved Markdown
        conn.execute("""
            CREATE VIRTUAL TABLE IF NOT EXISTS pages_fts USING fts5(
                search_title,
                url,
                title,
                content,
                content='pages',
                content_rowid='id'
            )
        """)
        # Triggers keep the FTS5 index automatically synced whenever rows are added
        conn.execute("""
            CREATE TRIGGER IF NOT EXISTS pages_ai AFTER INSERT ON pages BEGIN
                INSERT INTO pages_fts(rowid, search_title, url, title, content)
                VALUES (new.id, new.search_title, new.url, new.title, new.content);
            END;
        """)
        conn.commit()


def save_pages_to_db(search_title: str, pages: List[Dict[str, Any]], db_path: str = DB_PATH) -> str:
    """
    Offloads fetched Markdown into SQLite and returns a short receipt string.
    The LLM only sees this receipt—never the raw 30,000+ characters of Markdown.
    """
    now = datetime.now(timezone.utc).isoformat()
    saved_count = 0

    with sqlite3.connect(db_path) as conn:
        for p in pages:
            content = p.get("content", "")
            if not content:
                continue
            conn.execute(
                """
                INSERT INTO pages (search_title, site, url, title, content, fetched_at)
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    search_title,
                    p.get("site", ""),
                    p.get("url", ""),
                    p.get("title", "Untitled"),
                    content,
                    now,
                ),
            )
            saved_count += 1
        conn.commit()

    return (
        f"Saved {saved_count} page(s) in SQLite under search_title='{search_title}'. "
        f"Use query_saved_search(search_title, keyword) to read specific details."
    )


def query_saved_search(search_title: str, keyword: str, limit: int = 4) -> str:
    """
    TOOL FOR QWEN 3.5 4B:
    Searches the FTS5 index for `keyword` inside a specific `search_title` and
    returns only compact matching snippets (~30 tokens around each match).
    """
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.execute(
            """
            SELECT
                url,
                title,
                snippet(pages_fts, 3, '[', ']', ' ... ', 30) AS excerpt
            FROM pages_fts
            WHERE search_title = ? AND pages_fts MATCH ?
            ORDER BY rank
            LIMIT ?
            """,
            (search_title, keyword, limit),
        )
        rows = cursor.fetchall()

    if not rows:
        return f"No matches found for keyword '{keyword}' under search_title='{search_title}'."

    formatted = []
    for url, title, excerpt in rows:
        formatted.append(f"- Title: {title}\n  URL: {url}\n  Match: {excerpt}")
    return "\n\n".join(formatted)


# ==============================================================================
# BLOCK 3: KEENABLE API WRAPPERS (SEARCH + BATCH FETCH)
# What it does:
#   Provides two clean Python functions that talk to Keenable's REST API:
#   1. `keenable_search`: Lightweight "See-First" recon (short snippets only).
#   2. `keenable_fetch_batch`: Reads multiple URLs in a single POST /v1/fetch call.
# How it is implemented:
#   Uses `POST /v1/search` with `snippet_max_length=250` to keep recon tiny,
#   and `POST /v1/fetch` (which supports up to 20 URLs per request) to fetch
#   clean Markdown from Keenable's web index.
# ==============================================================================
def keenable_search(query: str, site: str = "", max_results: int = 3) -> List[Dict[str, Any]]:
    """Runs a lightweight Keenable search returning URLs, titles, and short snippets."""
    payload: Dict[str, Any] = {
        "query": query,
        "max_results": max_results,
        "snippet_max_length": 250,   # Keep snippets compact for the 4B model
        "mode": "realtime",
    }
    if site:
        payload["site"] = site

    resp = httpx.post(
        f"{KEENABLE_BASE_URL}/v1/search",
        headers={"X-API-Key": KEENABLE_API_KEY, "Content-Type": "application/json"},
        json=payload,
        timeout=25.0,
    )
    resp.raise_for_status()
    return resp.json().get("results", [])


def keenable_fetch_batch(urls: List[str], max_chars: int = 15000) -> List[Dict[str, Any]]:
    """Fetches up to 20 URLs in one POST /v1/fetch call as clean Markdown."""
    if not urls:
        return []

    resp = httpx.post(
        f"{KEENABLE_BASE_URL}/v1/fetch",
        headers={"X-API-Key": KEENABLE_API_KEY, "Content-Type": "application/json"},
        json={
            "urls": urls[:20],
            "content": {"max_chars": max_chars},
            # Omit `retrieval.max_age_hours` to use Keenable's fast indexed copy;
            # set `"retrieval": {"max_age_hours": 0}` if you need a live fetch.
        },
        timeout=35.0,
    )
    resp.raise_for_status()
    return resp.json().get("results", [])


# ==============================================================================
# BLOCK 4: LANGGRAPH STATE & SEQUENTIAL NODES
# What it does:
#   Defines the shared state dictionary passed between nodes and implements
#   the 4 stages of the pipeline:
#     1. `plan_node`             -> Qwen breaks down the prompt into a search plan
#     2. `see_first_node`        -> Keenable Search (snippets) + Qwen triage decision
#     3. `fetch_and_offload_node`-> Keenable Batch Fetch -> Save to SQLite FTS5
#     4. `db_analyzer_node`      -> Qwen queries SQLite via tool -> Final Answer
# How it is implemented:
#   Each node takes `AgentState`, performs one focused job, and returns only
#   the keys it updated.
# ==============================================================================
class AgentState(TypedDict):
    user_query: str
    search_title: str
    search_query: str
    target_sites: List[str]
    recon_results: List[Dict[str, Any]]
    proceed_to_fetch: bool
    selected_urls: List[str]
    db_receipt: str
    final_answer: str


def plan_node(state: AgentState) -> Dict[str, Any]:
    """
    NODE 1 (PLANNER):
    Uses Qwen 3.5 4B in strict JSON mode to turn the user's request into a
    clean search query, a short `search_title` for SQLite, and target domains.
    """
    res = ollama.chat(
        model=MODEL_NAME,
        format="json",
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a search planner. Convert the user's request into JSON with exact keys:\n"
                    '- "search_title": short snake_case label for DB storage (e.g. "red_shirt_men")\n'
                    '- "search_query": concise search string\n'
                    '- "target_sites": list of domains (e.g. ["amazon.in", "flipkart.com"]) or [] if general'
                ),
            },
            {"role": "user", "content": state["user_query"]},
        ],
    )
    plan = json.loads(res.message.content)
    print(f"[1. PLANNER] Plan: {plan}")
    return {
        "search_title": plan.get("search_title", "general_search"),
        "search_query": plan.get("search_query", state["user_query"]),
        "target_sites": plan.get("target_sites", []),
    }


def see_first_node(state: AgentState) -> Dict[str, Any]:
    """
    NODE 2 (SEE-FIRST / RECON):
    1. Calls Keenable Search to get ranked URLs + short snippets only.
    2. Asks Qwen 3.5 4B to inspect the snippets and decide:
       - Are the results normal and relevant?
       - Can we answer already from snippets, or should we fetch full pages?
    """
    sites = state["target_sites"] or [""]
    all_results = []
    for site in sites:
        hits = keenable_search(state["search_query"], site=site, max_results=2)
        for h in hits:
            h["site"] = site
            all_results.append(h)

    # Compact view of snippets for Qwen to inspect
    preview = [
        {"url": r.get("url"), "title": r.get("title"), "snippet": r.get("snippet", "")}
        for r in all_results
    ]

    res = ollama.chat(
        model=MODEL_NAME,
        format="json",
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a triage agent. Inspect the search snippets and return JSON with:\n"
                    '- "proceed_to_fetch": true if we need full page details (prices/specs/reviews), '
                    "false if snippets already answer the user or results are irrelevant.\n"
                    '- "selected_urls": list of up to 3 best URLs to fetch from the provided results.\n'
                    '- "reason": 1-sentence explanation.'
                ),
            },
            {
                "role": "user",
                "content": f"User query: {state['user_query']}\n\nSearch snippets:\n{json.dumps(preview, indent=2)}",
            },
        ],
    )
    decision = json.loads(res.message.content)
    print(f"[2. SEE-FIRST] Decision: proceed={decision.get('proceed_to_fetch')} | Reason: {decision.get('reason')}")
    return {
        "recon_results": all_results,
        "proceed_to_fetch": bool(decision.get("proceed_to_fetch", False)),
        "selected_urls": decision.get("selected_urls", [])[:3],
    }


def fetch_and_offload_node(state: AgentState) -> Dict[str, Any]:
    """
    NODE 3 (FETCH & OFFLOAD TO SQLITE):
    Fetches the selected URLs in one Keenable batch call, saves the full Markdown
    into SQLite + FTS5 under `search_title`, and returns only a tiny receipt.
    """
    urls = state["selected_urls"]
    print(f"[3. FETCH & OFFLOAD] Batch-fetching {len(urls)} URL(s)...")
    pages = keenable_fetch_batch(urls, max_chars=15000)

    receipt = save_pages_to_db(state["search_title"], pages)
    print(f"[3. FETCH & OFFLOAD] {receipt}")
    return {"db_receipt": receipt}


def db_analyzer_node(state: AgentState) -> Dict[str, Any]:
    """
    NODE 4 (DB-READER & ANALYZER):
    Gives Qwen 3.5 4B ONLY the `query_saved_search` tool so it can pull targeted
    chunks from SQLite (e.g., 'price', 'rating', 'cotton') without overflowing
    its context window, then synthesizes the final answer.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research analyst. Full webpage text has been saved to SQLite. "
                "Use the `query_saved_search` tool with specific single-word keywords "
                "(like 'price', 'rating', 'cotton', 'size') to retrieve relevant snippets, "
                "then answer the user's question concisely."
            ),
        },
        {
            "role": "user",
            "content": (
                f"User question: {state['user_query']}\n"
                f"Database status: {state['db_receipt']}\n"
                f"search_title to query: {state['search_title']}"
            ),
        },
    ]

    # Small bounded tool-calling loop (max 3 DB lookups)
    for step in range(3):
        res = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            tools=[query_saved_search],   # Ollama auto-generates schema from the Python function
            options=OLLAMA_OPTIONS,
            keep_alive=0,
        )
        messages.append(res.message)

        # If Qwen didn't request a tool call, it has produced the final answer
        if not res.message.tool_calls:
            return {"final_answer": res.message.content}

        # Execute each requested DB lookup and feed the compact snippet back
        for call in res.message.tool_calls:
            args = call.function.arguments
            print(f"[4. DB-ANALYZER] Step {step + 1} -> query_saved_search({args})")
            tool_output = query_saved_search(
                search_title=args.get("search_title", state["search_title"]),
                keyword=args.get("keyword", ""),
            )
            messages.append({"role": "tool", "content": tool_output})

    # Final synthesis call with tools DISABLED so the model must output its answer
    final_res = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
    )
    return {"final_answer": final_res.message.content}


def answer_from_snippets_node(state: AgentState) -> Dict[str, Any]:
    """
    FALLBACK NODE (EARLY EXIT):
    Runs only if `see_first_node` decided fetching full pages was unnecessary.
    """
    res = ollama.chat(
        model=MODEL_NAME,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {"role": "system", "content": "Answer the user concisely using ONLY the provided search snippets."},
            {
                "role": "user",
                "content": f"Query: {state['user_query']}\n\nSnippets:\n{json.dumps(state['recon_results'], indent=2)}",
            },
        ],
    )
    return {"final_answer": res.message.content}


# ==============================================================================
# BLOCK 5: BUILD & COMPILE THE LANGGRAPH WORKFLOW
# What it does:
#   Wires the nodes into a sequential graph with one conditional branch after
#   `see_first` (either proceed to Fetch + SQLite + DB Analyzer, or exit early).
# How it is implemented:
#   Uses `add_conditional_edges` on `see_first` to route dynamically based on
#   `state["proceed_to_fetch"]`.
# ==============================================================================
def route_after_see_first(state: AgentState) -> str:
    if state["proceed_to_fetch"] and state["selected_urls"]:
        return "fetch_and_offload"
    return "answer_from_snippets"


init_db()  # Ensure SQLite tables and FTS5 index exist

workflow = StateGraph(AgentState)
workflow.add_node("plan", plan_node)
workflow.add_node("see_first", see_first_node)
workflow.add_node("fetch_and_offload", fetch_and_offload_node)
workflow.add_node("db_analyzer", db_analyzer_node)
workflow.add_node("answer_from_snippets", answer_from_snippets_node)

workflow.add_edge(START, "plan")
workflow.add_edge("plan", "see_first")
workflow.add_conditional_edges(
    "see_first",
    route_after_see_first,
    {
        "fetch_and_offload": "fetch_and_offload",
        "answer_from_snippets": "answer_from_snippets",
    },
)
workflow.add_edge("fetch_and_offload", "db_analyzer")
workflow.add_edge("db_analyzer", END)
workflow.add_edge("answer_from_snippets", END)

agent = workflow.compile()


# ==============================================================================
# BLOCK 6: RUN THE PIPELINE
# What it does:
#   Executes the full compiled graph for a user prompt and prints the final answer.
# ==============================================================================
result = agent.invoke({
    "user_query": "Find a red cotton shirt for men on amazon.in and flipkart.com and compare prices and ratings",
    "search_title": "",
    "search_query": "",
    "target_sites": [],
    "recon_results": [],
    "proceed_to_fetch": False,
    "selected_urls": [],
    "db_receipt": "",
    "final_answer": "",
})

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_answer"])

[1. PLANNER] Plan: {'search_title': 'red_cotton_shirt_men', 'search_query': 'red cotton shirt men prices ratings comparison', 'target_sites': ['amazon.in', 'flipkart.com']}
[2. SEE-FIRST] Decision: proceed=True | Reason: Full page details are required to verify the missing price for the Amazon product in the snippets and to ensure accurate comparison across both platforms.
[3. FETCH & OFFLOAD] Batch-fetching 3 URL(s)...
[3. FETCH & OFFLOAD] Saved 3 page(s) in SQLite under search_title='red_cotton_shirt_men'. Use query_saved_search(search_title, keyword) to read specific details.
[4. DB-ANALYZER] Step 1 -> query_saved_search({'search_title': 'red_cotton_shirt_men', 'keyword': 'price', 'limit': 10})
[4. DB-ANALYZER] Step 2 -> query_saved_search({'search_title': 'red_cotton_shirt_men', 'keyword': 'rating', 'limit': 10})

FINAL ANSWER
## Comparison: Red Cotton Shirts for Men (Amazon.in & Flipkart.com)

| Product | Platform | Rating | Price | Key Details |
|---------|----------|--------|---

In [1]:
# ==============================================================================
# BLOCK 1: IMPORTS & CONFIGURATION
# What it does:
#   Centralizes API keys, model names, SQLite path, screenshot directory, and
#   Playwright settings for both Jupyter testing and FastAPI/backend serving.
# How it is implemented:
#   Uses async Playwright + async LangGraph so it works seamlessly in both
#   Jupyter Notebooks and async web servers (FastAPI/Uvicorn).
# ==============================================================================
import os
import re
import json
import sqlite3
from pathlib import Path
from datetime import datetime, timezone
from typing import TypedDict, List, Dict, Any, AsyncGenerator

import httpx
import ollama
from playwright.async_api import async_playwright
from langgraph.graph import StateGraph, START, END

# Set your key in your environment or .env file first
os.environ["KEENABLE_API_KEY"] = "keen_4yd5wfrkvg_84c5b059d5f8bef0135c8c07fa556c3347a9e75daa19239e486253ba4206d549"
API_KEY = os.environ["KEENABLE_API_KEY"]

KEENABLE_API_KEY = os.environ["KEENABLE_API_KEY"]
KEENABLE_BASE_URL = "https://api.keenable.ai"

TEXT_MODEL = "qwen3.5:4b"               # Planner, See-First triage, & DB Analyzer
VL_MODEL = "qwen3-vl:4b-instruct"       # Visual verification on winning URLs
DB_PATH = "agent_context.db"
SCREENSHOT_DIR = Path("screenshots")    # Serve this folder as static files in your frontend
SCREENSHOT_DIR.mkdir(exist_ok=True)

BROWSER = "chrome"                      # "chromium", "firefox", "webkit", "chrome", "msedge"
HEADLESS = True                         # True for backend serving; False to watch locally
OLLAMA_OPTIONS = {}                     # Set to {"num_gpu": 0} if Windows CUDA crashes


# ==============================================================================
# BLOCK 2: SQLITE + FTS5 CONTEXT-OFFLOADING DATABASE
# What it does:
#   Stores full Keenable Markdown under `search_title` and provides a compact
#   FTS5 keyword-search tool (`query_saved_search`) for Qwen 3.5 4B.
# Fixes included:
#   1. Clears old rows for the same `search_title` and rebuilds FTS5 so previous
#      runs never leak into new results.
#   2. Quotes/sanitizes FTS5 keywords so terms like "RTX-3050" or "1.5k" never
#      trigger SQLite FTS5 syntax errors.
# ==============================================================================
def init_db(db_path: str = DB_PATH) -> None:
    with sqlite3.connect(db_path) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                search_title TEXT NOT NULL,
                site TEXT,
                url TEXT NOT NULL,
                title TEXT,
                content TEXT NOT NULL,
                fetched_at TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE VIRTUAL TABLE IF NOT EXISTS pages_fts USING fts5(
                search_title, url, title, content,
                content='pages', content_rowid='id'
            )
        """)
        conn.execute("""
            CREATE TRIGGER IF NOT EXISTS pages_ai AFTER INSERT ON pages BEGIN
                INSERT INTO pages_fts(rowid, search_title, url, title, content)
                VALUES (new.id, new.search_title, new.url, new.title, new.content);
            END;
        """)
        conn.commit()


def save_pages_to_db(search_title: str, pages: List[Dict[str, Any]], db_path: str = DB_PATH) -> str:
    now = datetime.now(timezone.utc).isoformat()
    saved_count = 0
    with sqlite3.connect(db_path) as conn:
        # Remove stale rows for this search_title and rebuild FTS5 cleanly
        conn.execute("DELETE FROM pages WHERE search_title = ?", (search_title,))
        conn.execute("INSERT INTO pages_fts(pages_fts) VALUES('rebuild')")

        for p in pages:
            content = p.get("content", "")
            if not content:
                continue
            conn.execute(
                "INSERT INTO pages (search_title, site, url, title, content, fetched_at) VALUES (?, ?, ?, ?, ?, ?)",
                (search_title, p.get("site", ""), p.get("url", ""), p.get("title", "Untitled"), content, now),
            )
            saved_count += 1
        conn.commit()

    return (
        f"Saved {saved_count} fresh page(s) in SQLite under search_title='{search_title}'. "
        f"Use query_saved_search(search_title, keyword) to read specific details."
    )


def _sanitize_fts_keyword(keyword: str) -> str:
    """Safely quotes a keyword for SQLite FTS5 MATCH to prevent syntax errors."""
    cleaned = re.sub(r'["\'*():^]', " ", str(keyword or "")).strip()
    if not cleaned:
        return ""
    return f'"{cleaned}"'


def query_saved_search(search_title: str, keyword: str, limit: int = 8) -> str:
    """TOOL FOR QWEN 3.5 4B: Returns compact FTS5 snippets matching `keyword`."""
    fts_term = _sanitize_fts_keyword(keyword)
    if not fts_term:
        return "Empty keyword provided. Pass a specific word like 'price', 'rating', 'RTX', or 'leather'."

    try:
        with sqlite3.connect(DB_PATH) as conn:
            cursor = conn.execute(
                """
                SELECT url, title, snippet(pages_fts, 3, '[', ']', ' ... ', 30) AS excerpt
                FROM pages_fts
                WHERE search_title = ? AND pages_fts MATCH ?
                ORDER BY rank
                LIMIT ?
                """,
                (search_title, fts_term, limit),
            )
            rows = cursor.fetchall()
    except sqlite3.OperationalError as e:
        return f"FTS query error for '{keyword}': {e}. Try a single alphanumeric keyword."

    if not rows:
        return f"No matches for '{keyword}' under search_title='{search_title}'."
    return "\n\n".join(f"- Title: {t}\n  URL: {u}\n  Match: {e}" for u, t, e in rows)


# ==============================================================================
# BLOCK 3: FAST RESEARCH TOOLSET (KEENABLE SEARCH + BATCH FETCH)
# What it does:
#   Handles fast, text-only discovery and Markdown retrieval without launching
#   a browser.
# ==============================================================================
async def keenable_search(query: str, site: str = "", max_results: int = 3) -> List[Dict[str, Any]]:
    payload: Dict[str, Any] = {
        "query": query,
        "max_results": max_results,
        "snippet_max_length": 250,
        "mode": "realtime",
    }
    if site:
        payload["site"] = site

    async with httpx.AsyncClient(timeout=25.0) as client:
        resp = await client.post(
            f"{KEENABLE_BASE_URL}/v1/search",
            headers={"X-API-Key": KEENABLE_API_KEY, "Content-Type": "application/json"},
            json=payload,
        )
        resp.raise_for_status()
        return resp.json().get("results", [])


async def keenable_fetch_batch(urls: List[str], max_chars: int = 15000) -> List[Dict[str, Any]]:
    if not urls:
        return []
    async with httpx.AsyncClient(timeout=35.0) as client:
        resp = await client.post(
            f"{KEENABLE_BASE_URL}/v1/fetch",
            headers={"X-API-Key": KEENABLE_API_KEY, "Content-Type": "application/json"},
            json={"urls": urls[:20], "content": {"max_chars": max_chars}},
        )
        resp.raise_for_status()
        return resp.json().get("results", [])


# ==============================================================================
# BLOCK 4: VISUAL & PRESENTATION TOOLSET (PLAYWRIGHT + QWEN-VL)
# What it does:
#   Runs ONLY on the final winning product URLs to:
#   1. Extract the primary product image URL (Amazon #landingImage / Flipkart / og:image)
#   2. Capture a clean viewport screenshot for your frontend UI
#   3. Run Qwen-VL on the product image so the model visually confirms the item
# ==============================================================================
IMAGE_EXTRACT_JS = """() => {
    // 1. Amazon & Flipkart primary product image selectors
    const primarySelectors = [
        '#landingImage',
        'img[data-old-hires]',
        'img.s-image',
        'img[src*="rukminim"]',
        'img[src*="m.media-amazon.com/images/I/"]'
    ];
    for (const sel of primarySelectors) {
        const el = document.querySelector(sel);
        if (el) {
            const src = el.getAttribute('data-old-hires') || el.currentSrc || el.src;
            if (src && src.startsWith('http')) return src;
        }
    }

    // 2. OpenGraph meta image fallback
    const og = document.querySelector('meta[property="og:image"]');
    if (og && og.content && og.content.startsWith('http')) return og.content;

    // 3. Largest visible image fallback
    const imgs = Array.from(document.querySelectorAll('img'))
        .map(i => ({ src: i.currentSrc || i.src, area: (i.naturalWidth || 0) * (i.naturalHeight || 0) }))
        .filter(i => i.src && i.src.startsWith('http') && i.area >= 40000)
        .sort((a, b) => b.area - a.area);
    return imgs.length ? imgs[0].src : '';
}"""


async def capture_product_visuals(urls: List[str], search_title: str) -> Dict[str, Dict[str, str]]:
    """Visits winning URLs to grab the main product image URL + a UI screenshot."""
    visuals: Dict[str, Dict[str, str]] = {}
    if not urls:
        return visuals

    stealth_args = ["--disable-blink-features=AutomationControlled"]
    async with async_playwright() as p:
        if BROWSER in ("chrome", "msedge"):
            browser = await p.chromium.launch(channel=BROWSER, headless=HEADLESS, args=stealth_args)
        elif BROWSER == "chromium":
            browser = await p.chromium.launch(headless=HEADLESS, args=stealth_args)
        else:
            browser = await getattr(p, BROWSER).launch(headless=HEADLESS)

        context = await browser.new_context(viewport={"width": 1280, "height": 800}, locale="en-IN")
        page = await context.new_page()

        for idx, url in enumerate(urls, start=1):
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=20000)
                await page.wait_for_timeout(2000)

                image_url = await page.evaluate(IMAGE_EXTRACT_JS)

                safe_name = re.sub(r"[^a-zA-Z0-9_-]", "_", search_title)
                shot_path = SCREENSHOT_DIR / f"{safe_name}_{idx}.png"
                await page.screenshot(path=str(shot_path), full_page=False)

                visuals[url] = {
                    "image_url": image_url,
                    "screenshot_path": shot_path.as_posix(),
                }
            except Exception as e:
                visuals[url] = {"image_url": "", "screenshot_path": "", "error": str(e)}

        await browser.close()
    return visuals


async def verify_image_with_vl(image_url: str, user_query: str, title: str) -> Dict[str, Any]:
    """Downloads the product image and asks Qwen-VL to visually verify it."""
    if not image_url:
        return {"vl_verified": False, "vl_note": "No product image found to verify."}

    try:
        async with httpx.AsyncClient(timeout=15.0, follow_redirects=True) as client:
            resp = await client.get(image_url)
            resp.raise_for_status()
            img_bytes = resp.content

        res = ollama.chat(
            model=VL_MODEL,
            format="json",
            options=OLLAMA_OPTIONS,
            keep_alive=0,
            messages=[{
                "role": "user",
                "content": (
                    f"User requested: {user_query}\nProduct title: {title}\n"
                    "Inspect this product image and return JSON with exact keys:\n"
                    '- "vl_verified": true if the image matches the product title and fits the user request, else false\n'
                    '- "vl_note": brief 1-sentence visual description of what you see'
                ),
                "images": [img_bytes],
            }],
        )
        return json.loads(res.message.content)
    except Exception as e:
        return {"vl_verified": False, "vl_note": f"VL check skipped: {e}"}


# ==============================================================================
# BLOCK 5: LANGGRAPH STATE & NODES
# What it does:
#   Defines the shared state and the 5 pipeline nodes:
#     1. plan              -> Splits multi-item prompts into `search_queries` + `constraints`
#     2. see_first         -> Keenable Search per item query + Qwen triage
#     3. fetch_and_offload -> Keenable Batch Fetch -> Fresh SQLite FTS5
#     4. db_analyzer       -> Qwen queries SQLite -> structured `products` + `summary_markdown`
#     5. visual_enrich     -> Playwright + Qwen-VL attaches images, screenshots & VL badges
# ==============================================================================
class AgentState(TypedDict):
    user_query: str
    search_title: str
    search_query: str
    search_queries: List[str]            # One query per requested item
    constraints: str                     # Budget, preferred colors, or required specs
    target_sites: List[str]
    recon_results: List[Dict[str, Any]]
    proceed_to_fetch: bool
    selected_urls: List[str]
    db_receipt: str
    products: List[Dict[str, Any]]       # Structured cards for your frontend UI
    summary_markdown: str                # Final comparison text for your frontend UI


async def plan_node(state: AgentState) -> Dict[str, Any]:
    res = ollama.chat(
        model=TEXT_MODEL,
        format="json",
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "Convert the user's shopping request into JSON with exact keys:\n"
                    '- "search_title": short snake_case DB label (e.g. "formal_outfit_men", "leather_jacket_budget", "rtx_3050_laptop")\n'
                    '- "search_queries": list of concise search strings—ONE per distinct item requested '
                    '(e.g. ["white or pink formal shirt men", "formal pant men", "formal belt men", "formal shoes men"]). '
                    "Correct any spelling typos (e.g. 'nvdia 3050' -> 'NVIDIA RTX 3050 laptop').\n"
                    '- "target_sites": list of Indian shopping domains mentioned, or default to ["amazon.in", "flipkart.com"]\n'
                    '- "constraints": short string summarizing budget, preferred colors, or required specs '
                    '(e.g. "Budget: ₹1,500–₹3,000" or "GPU: NVIDIA RTX 3050, include full specs")'
                ),
            },
            {"role": "user", "content": state["user_query"]},
        ],
    )
    plan = json.loads(res.message.content)
    queries = plan.get("search_queries") or [plan.get("search_query", state["user_query"])]
    sites = plan.get("target_sites") or ["amazon.in", "flipkart.com"]
    return {
        "search_title": plan.get("search_title", "general_search"),
        "search_query": " | ".join(queries),
        "search_queries": queries,
        "target_sites": sites,
        "constraints": plan.get("constraints", ""),
    }


async def see_first_node(state: AgentState) -> Dict[str, Any]:
    sites = state["target_sites"] or ["amazon.in", "flipkart.com"]
    queries = state.get("search_queries") or [state["search_query"]]
    # Scale URL budget so multi-item outfits get coverage for every item
    max_urls = min(max(3, len(queries) * 2), 8)
    per_site_hits = 1 if len(queries) >= 3 else 2

    all_results = []
    for q in queries:
        for site in sites:
            hits = await keenable_search(q, site=site, max_results=per_site_hits)
            for h in hits:
                h["site"] = site
                h["item_query"] = q
                all_results.append(h)

    preview = [
        {"item": r.get("item_query"), "url": r.get("url"), "title": r.get("title"), "snippet": r.get("snippet", "")}
        for r in all_results
    ]
    res = ollama.chat(
        model=TEXT_MODEL,
        format="json",
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {
                "role": "system",
                "content": (
                    f"Inspect the search snippets and select up to {max_urls} best product page URLs "
                    "covering EVERY distinct item requested by the user (prefer individual product pages over category listings).\n"
                    "Return JSON with exact keys:\n"
                    '- "proceed_to_fetch": true if full page details (prices/specs/ratings) are needed, else false\n'
                    '- "selected_urls": list of best URLs copied verbatim from the provided results\n'
                    '- "reason": 1-sentence explanation'
                ),
            },
            {
                "role": "user",
                "content": (
                    f"User query: {state['user_query']}\n"
                    f"Constraints: {state.get('constraints', '')}\n\n"
                    f"Snippets:\n{json.dumps(preview, indent=2)}"
                ),
            },
        ],
    )
    decision = json.loads(res.message.content)
    return {
        "recon_results": all_results,
        "proceed_to_fetch": bool(decision.get("proceed_to_fetch", False)),
        "selected_urls": decision.get("selected_urls", [])[:max_urls],
    }


async def fetch_and_offload_node(state: AgentState) -> Dict[str, Any]:
    pages = await keenable_fetch_batch(state["selected_urls"], max_chars=15000)
    receipt = save_pages_to_db(state["search_title"], pages)
    return {"db_receipt": receipt}


async def db_analyzer_node(state: AgentState) -> Dict[str, Any]:
    """
    Queries SQLite via `query_saved_search`, then produces BOTH:
    - `products`: structured list of product dicts for frontend cards
    - `summary_markdown`: comparison table & recommendation honoring constraints
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research analyst. Full webpage text is saved in SQLite. "
                "Call `query_saved_search` with single-word keywords relevant to the request "
                "(e.g. 'price', 'rating', 'RAM', 'SSD', 'RTX', 'leather', 'shirt', 'shoes', 'belt') "
                "to gather prices, ratings, and specs, then stop calling tools."
            ),
        },
        {
            "role": "user",
            "content": (
                f"User question: {state['user_query']}\n"
                f"Items searched: {state.get('search_queries', [])}\n"
                f"Constraints: {state.get('constraints', 'None')}\n"
                f"DB status: {state['db_receipt']}\n"
                f"search_title: {state['search_title']}"
            ),
        },
    ]

    # Step A: Bounded DB lookup loop (max 4 tool calls for multi-item / spec-heavy queries)
    for _ in range(4):
        res = ollama.chat(
            model=TEXT_MODEL,
            messages=messages,
            tools=[query_saved_search],
            options=OLLAMA_OPTIONS,
            keep_alive=0,
        )
        messages.append(res.message)
        if not res.message.tool_calls:
            break
        for call in res.message.tool_calls:
            args = call.function.arguments
            out = query_saved_search(
                search_title=args.get("search_title", state["search_title"]),
                keyword=args.get("keyword", ""),
            )
            messages.append({"role": "tool", "content": out})

    # Step B: Final synthesis with tools DISABLED -> strict JSON for frontend
    messages.append({
        "role": "user",
        "content": (
            "Now return ONLY valid JSON with two keys:\n"
            '1. "products": list of objects for the best matching products (strictly respect budget/spec constraints where possible), '
            'each with keys: "category", "title", "platform", "price", "rating", "url", "key_details"\n'
            '2. "summary_markdown": concise Markdown table comparing the items/specs and a clear recommendation.'
        ),
    })
    final_res = ollama.chat(
        model=TEXT_MODEL,
        format="json",
        messages=messages,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
    )
    parsed = json.loads(final_res.message.content)
    return {
        "products": parsed.get("products", []),
        "summary_markdown": parsed.get("summary_markdown", ""),
    }


async def visual_enrich_node(state: AgentState) -> Dict[str, Any]:
    """
    NODE 5 (VISUAL & PRESENTATION LAYER):
    Runs Playwright + Qwen-VL ONLY on the final product URLs so the frontend
    can display real product photos/screenshots and VL verification badges.
    """
    products = state.get("products", [])
    urls = [p["url"] for p in products if p.get("url")] or state.get("selected_urls", [])

    visuals_map = await capture_product_visuals(urls, state["search_title"])

    enriched = []
    for p in products:
        url = p.get("url", "")
        vis = visuals_map.get(url, {})
        img_url = vis.get("image_url", "")
        vl = await verify_image_with_vl(img_url, state["user_query"], p.get("title", ""))

        enriched.append({
            **p,
            "image_url": img_url,
            "screenshot_path": vis.get("screenshot_path", ""),
            "vl_verified": bool(vl.get("vl_verified", False)),
            "vl_note": vl.get("vl_note", ""),
        })

    return {"products": enriched}


async def answer_from_snippets_node(state: AgentState) -> Dict[str, Any]:
    res = ollama.chat(
        model=TEXT_MODEL,
        options=OLLAMA_OPTIONS,
        keep_alive=0,
        messages=[
            {"role": "system", "content": "Answer concisely using ONLY the provided search snippets."},
            {"role": "user", "content": f"Query: {state['user_query']}\n\nSnippets:\n{json.dumps(state['recon_results'], indent=2)}"},
        ],
    )
    return {"products": [], "summary_markdown": res.message.content}


# ==============================================================================
# BLOCK 6: COMPILE GRAPH & FRONTEND STREAMING INTERFACE
# ==============================================================================
def route_after_see_first(state: AgentState) -> str:
    return "fetch_and_offload" if (state["proceed_to_fetch"] and state["selected_urls"]) else "answer_from_snippets"


init_db()

workflow = StateGraph(AgentState)
workflow.add_node("plan", plan_node)
workflow.add_node("see_first", see_first_node)
workflow.add_node("fetch_and_offload", fetch_and_offload_node)
workflow.add_node("db_analyzer", db_analyzer_node)
workflow.add_node("visual_enrich", visual_enrich_node)
workflow.add_node("answer_from_snippets", answer_from_snippets_node)

workflow.add_edge(START, "plan")
workflow.add_edge("plan", "see_first")
workflow.add_conditional_edges("see_first", route_after_see_first, {
    "fetch_and_offload": "fetch_and_offload",
    "answer_from_snippets": "answer_from_snippets",
})
workflow.add_edge("fetch_and_offload", "db_analyzer")
workflow.add_edge("db_analyzer", "visual_enrich")
workflow.add_edge("visual_enrich", END)
workflow.add_edge("answer_from_snippets", END)

agent = workflow.compile()


async def run_pipeline_stream(user_query: str) -> AsyncGenerator[Dict[str, Any], None]:
    """Yields real-time step events + final UI payload for your separate frontend."""
    initial_state: AgentState = {
        "user_query": user_query,
        "search_title": "",
        "search_query": "",
        "search_queries": [],
        "constraints": "",
        "target_sites": [],
        "recon_results": [],
        "proceed_to_fetch": False,
        "selected_urls": [],
        "db_receipt": "",
        "products": [],
        "summary_markdown": "",
    }

    final_products: List[Dict[str, Any]] = []
    final_summary = ""

    async for chunk in agent.astream(initial_state, stream_mode="updates"):
        for node_name, update in chunk.items():
            if "products" in update:
                final_products = update["products"]
            if "summary_markdown" in update:
                final_summary = update["summary_markdown"]

            yield {
                "type": "step",
                "node": node_name,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "data": update,
            }

    yield {
        "type": "complete",
        "products": final_products,
        "summary_markdown": final_summary,
    }


# ==============================================================================
# BLOCK 7: TEST RUNNER HELPER FOR YOUR 3 PROMPTS
# ==============================================================================
async def test_prompt(prompt: str):
    print("=" * 70)
    print(f"TESTING PROMPT: {prompt}")
    print("=" * 70)
    async for event in run_pipeline_stream(prompt):
        if event["type"] == "step":
            node = event["node"]
            data = event["data"]
            if node == "plan":
                print(f"[plan] queries={data.get('search_queries')} | constraints={data.get('constraints')}")
            elif node == "see_first":
                print(f"[see_first] proceed={data.get('proceed_to_fetch')} | urls={len(data.get('selected_urls', []))}")
            else:
                print(f"[{node}] updated -> {list(data.keys())}")
        elif event["type"] == "complete":
            print("\n=== FINAL UI PAYLOAD (PRODUCTS) ===")
            print(json.dumps(event["products"], indent=2))
            print("\n=== SUMMARY MARKDOWN ===")
            print(event["summary_markdown"])

In [6]:
# Prompt 1: Multi-item formal outfit
await test_prompt("i need formal shirts (white or pink preferable), pant, belt, shoes")

TESTING PROMPT: i need formal shirts (white or pink preferable), pant, belt, shoes
[plan] queries=['white pink formal shirts men', 'formal pants men', 'formal belt men', 'formal shoes men'] | constraints=Shirt Colors: White/Pink
[see_first] proceed=True | urls=8
[fetch_and_offload] updated -> ['db_receipt']
[db_analyzer] updated -> ['products', 'summary_markdown']
[visual_enrich] updated -> ['products']

=== FINAL UI PAYLOAD (PRODUCTS) ===
[
  {
    "category": "formal_shirt",
    "title": "Raymond Men Striped Formal White, Pink Shirt",
    "platform": "Flipkart.com",
    "price": "INR TBC",
    "rating": 4.2,
    "url": "https://flipkart.com/raymond-men-striped-formal-white-pink-shirt/p/itmfdkdvbfy9fpha",
    "key_details": "White/Pink striped formal, Polyester material",
    "image_url": "https://rukminim2.flixcart.com/image/800/1070/jvmpci80/shirt/8/v/h/39-rmsz08860-r3-raymond-original-imafghw5qzdavpch.jpeg?q=80",
    "screenshot_path": "screenshots/formal_clothing_men_1.png",
    "

In [2]:
# Prompt 2: Budget-constrained leather jacket
await test_prompt("Need to buy a leather jacket within budget of 1.5k to 3k")

TESTING PROMPT: Need to buy a leather jacket within budget of 1.5k to 3k
[plan] queries=['leather jacket men under 3k', 'affordable leather jackets india'] | constraints=Budget: ₹1,500–₹3,000
[see_first] proceed=True | urls=4
[fetch_and_offload] updated -> ['db_receipt']
[db_analyzer] updated -> ['products', 'summary_markdown']
[visual_enrich] updated -> ['products']

=== FINAL UI PAYLOAD (PRODUCTS) ===
[
  {
    "category": "Clothing",
    "title": "EXOTIX Black Leather Jacket for Men - Collared Zip-Up Classic Style",
    "platform": "Amazon.in",
    "price": 755,
    "rating": 4.0,
    "url": "https://www.amazon.in/EXOTIX-Leather-Collared-Classic-Premium/dp/B0H1CQG4YS?psc=1&ref_=fplfs&smid=A2Z8ZGE8TYZ4SA",
    "key_details": "Crop Fit, Racing Sports Bomber Jacket, Regular Fit",
    "image_url": "https://m.media-amazon.com/images/I/51aa6X0QOmL._SL1280_.jpg",
    "screenshot_path": "screenshots/leather_jacket_budget_1.png",
    "vl_verified": true,
    "vl_note": "A black leather jacke